<a href="https://colab.research.google.com/github/arthur-gui-22/PROJETOS_DATA_SCIENCE-/blob/CODE-HTML/eda_exploratoria_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exploratory Data Analysis (EDA) — guia prático com estatística básica

Este notebook guia você por um fluxo de **Análise Exploratória de Dados** usando um dataset de vendas sintético.
Ele cobre princípios estatísticos básicos (média, mediana, desvio-padrão, quartis, correlação) e visualizações úteis.


## 1) Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

## 2) Carregar dados

In [ ]:
# Opção A: usar a planilha gerada (faça upload do arquivo no Colab e ajuste o caminho se necessário)
caminho_excel = 'planilha_eda_exemplo.xlsx'  # altere se necessário
df = pd.read_excel(caminho_excel, sheet_name='dados')

# Opção B: usar outro arquivo (CSV/Excel) do seu projeto
# df = pd.read_csv('seu_arquivo.csv')
# df = pd.read_excel('seu_arquivo.xlsx', sheet_name=0)

df.head()

## 3) Inspeção estrutural

In [ ]:
print('Formato (linhas, colunas):', df.shape)
print('\nTipos de dados:')
print(df.dtypes)

print('\nValores ausentes por coluna:')
print(df.isna().sum())

print('\nEstatísticas descritivas numéricas:')
display(df.describe().T)

print('\nEstatísticas descritivas categóricas:')
display(df.select_dtypes('object').describe().T)

## 4) Limpeza básica (opcional)

In [ ]:
# Exemplo: tratar duplicatas
duplicatas = df.duplicated().sum()
print(f'Duplicatas: {duplicatas}')
df = df.drop_duplicates()

# Exemplo: tratar valores ausentes numéricos (imputar mediana)
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Exemplo: converter datas (se necessário)
if 'data_venda' in df.columns:
    df['data_venda'] = pd.to_datetime(df['data_venda'], errors='coerce')

## 5) Distribuições e outliers

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in ['preco_unitario', 'quantidade', 'receita_liquida', 'desconto_pct']:
    if col in df.columns:
        fig = plt.figure(figsize=(6,4))
        plt.hist(df[col].dropna(), bins=30)
        plt.title(f'Histograma — {col}')
        plt.xlabel(col); plt.ylabel('Frequência')
        plt.show()

        fig = plt.figure(figsize=(4,5))
        plt.boxplot(df[col].dropna(), vert=True)
        plt.title(f'Boxplot — {col}')
        plt.ylabel(col)
        plt.show()

# IQR method example for outliers (em uma coluna)
col = 'receita_liquida'
if col in df.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5*IQR
    lim_sup = Q3 + 1.5*IQR
    outliers = df[(df[col] < lim_inf) | (df[col] > lim_sup)]
    print(f'Outliers em {col}: {outliers.shape[0]} linhas')

## 6) Tendência central e dispersão

In [ ]:
def resumo_basico(x):
    return pd.Series({
        'count': x.count(),
        'mean': x.mean(),
        'median': x.median(),
        'std': x.std(),
        'min': x.min(),
        'q1': x.quantile(0.25),
        'q3': x.quantile(0.75),
        'max': x.max(),
        'skew': x.skew(),
        'kurtosis': x.kurtosis()
    })

resumo = df.select_dtypes(include=[np.number]).apply(resumo_basico).T
resumo

## 7) Relacionamentos (correlação e scatter)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
corr_pearson = df[num_cols].corr(method='pearson')
corr_spearman = df[num_cols].corr(method='spearman')

print('Correlação de Pearson (numéricas):')
display(corr_pearson)

print('Correlação de Spearman (numéricas):')
display(corr_spearman)

# Scatter entre duas variáveis relevantes
if set(['preco_unitario','receita_liquida']).issubset(df.columns):
    plt.figure(figsize=(6,5))
    plt.scatter(df['preco_unitario'], df['receita_liquida'])
    plt.xlabel('preco_unitario'); plt.ylabel('receita_liquida')
    plt.title('Scatter: preço unitário vs. receita líquida')
    plt.show()

## 8) Agregações por grupo e tabelas cruzadas

In [ ]:
# Agregações por categoria e região
if set(['categoria','regiao']).issubset(df.columns):
    ag = (df
          .groupby(['categoria','regiao'], as_index=False)
          .agg(qtd=('quantidade','sum'),
               receita=('receita_liquida','sum'),
               preco_medio=('preco_unitario','mean'))
         )
    display(ag.sort_values('receita', ascending=False).head(10))

# Tabela cruzada: devolução x canal
if set(['devolvido','canal_venda']).issubset(df.columns):
    ct = pd.crosstab(df['devolvido'], df['canal_venda'], normalize='columns').round(3)
    display(ct)

## 9) Séries temporais (se aplicável)

In [ ]:
if 'data_venda' in df.columns:
    df['data_venda'] = pd.to_datetime(df['data_venda'], errors='coerce')
    diario = (df
              .set_index('data_venda')
              .resample('W')['receita_liquida']
              .sum()
             )
    diario.plot(figsize=(8,3))
    plt.title('Receita líquida — semanal')
    plt.xlabel('Semana'); plt.ylabel('Receita')
    plt.show()